In [14]:
# Prever a quantidade de massa consumida em função das variáveis de produção e dos equipamentos.

In [15]:
path_dados = "../datasets/"
df_producao = "IJ_046_production_process_data.csv"
df_equipamentos = "Equipment_Data_Prediction_Taking.csv"

In [16]:
import pandas as pd

# Ler os arquivos CSV em DataFrames
df_equipamentos = pd.read_csv(path_dados + df_equipamentos)
df_producao = pd.read_csv(path_dados + df_producao)

In [17]:
# Exibir as 5 primeiras linhas de cada DataFrame
print("DataFrame Equipamentos:")
print(df_equipamentos.head().to_markdown(index=False, numalign="left", stralign="left"))

print("\nDataFrame Produção:")
print(df_producao.head().to_markdown(index=False, numalign="left", stralign="left"))

# Exibir as colunas e seus tipos de cada DataFrame
print("\nInformações do DataFrame Equipamentos:")
print(df_equipamentos.info())

print("\nInformações do DataFrame Produção:")
print(df_producao.info())

DataFrame Equipamentos:
| EQUIPAMENTO   | DENOMINAÇÃO                | POSIÇÃO DO CANHÃO DE INJEÇÃO   | MODELO   | TIPO OBJETO (FAMÍLIA)   | ANO CONSTRUÇÃO   | DATA ENTRADA EM SERVIÇO   |
|:--------------|:---------------------------|:-------------------------------|:---------|:------------------------|:-----------------|:--------------------------|
| IJ-044        | INJETORA DE BORRACHA - REP | SUPERIOR                       | V17EY    | IRP1                    | 2001             | 2011-11-08                |
| IJ-046        | INJETORA DE BORRACHA - REP | SUPERIOR                       | V17EY    | IRP1                    | 2003             | 2011-10-24                |
| IJ-117        | INJETORA DE BORRACHA - REP | SUPERIOR                       | V17EY    | IRP1                    | 2004             | 2011-10-28                |
| IJ-118        | INJETORA DE BORRACHA - REP | SUPERIOR                       | V17EY    | IRP1                    | 2004             | 2011-10-26          

In [18]:
# Obter os valores únicos de cada coluna
unique_equipamentos = df_equipamentos["EQUIPAMENTO"].unique()
unique_cod_recurso = df_producao["Cód. Recurso"].unique()

# Verificar o número de valores únicos em cada coluna
if len(unique_equipamentos) > 50:
    # Se houver muitos valores únicos, calcular a frequência e exibir os 50 mais frequentes
    top_equipamentos = (
        df_equipamentos["EQUIPAMENTO"].value_counts().head(50).index.tolist()
    )
    print(f"Os 50 equipamentos mais frequentes são: {top_equipamentos}")
else:
    # Caso contrário, exibir todos os valores únicos
    print(f"Os valores únicos da coluna 'EQUIPAMENTO' são: {unique_equipamentos}")

if len(unique_cod_recurso) > 50:
    # Se houver muitos valores únicos, calcular a frequência e exibir os 50 mais frequentes
    top_cod_recurso = df_producao["Cód. Recurso"].value_counts().head(50).index.tolist()
    print(f"Os 50 recursos mais frequentes são: {top_cod_recurso}")
else:
    # Caso contrário, exibir todos os valores únicos
    print(f"Os valores únicos da coluna 'Cód. Recurso' são: {unique_cod_recurso}")

Os valores únicos da coluna 'EQUIPAMENTO' são: ['IJ-044' 'IJ-046' 'IJ-117' 'IJ-118' 'IJ-119' 'IJ-120' 'IJ-121' 'IJ-122'
 'IJ-123' 'IJ-124' 'IJ-125' 'IJ-129' 'IJ-130' 'IJ-131' 'IJ-132' 'IJ-133'
 'IJ-134' 'IJ-135' 'IJ-136' 'IJ-137' 'IJ-138' 'IJ-139' 'IJ-151' 'IJ-152'
 'IJ-155' 'IJ-156' 'IJ-164' nan]
Os valores únicos da coluna 'Cód. Recurso' são: ['IJ-046']


In [19]:
# Vejo que os dados de produção fornecidos são apenas para o recurso 'IJ-046'. Portanto, para construir um modelo preditivo, irei focar nos dados relacionados a este recurso.

# Filtrar df_equipamentos para incluir apenas as linhas em que a coluna "EQUIPAMENTO" é igual a "IJ-046"
df_equipamentos = df_equipamentos[df_equipamentos["EQUIPAMENTO"] == "IJ-046"]

# Exibir as 5 primeiras linhas do DataFrame filtrado
print("DataFrame Equipamentos (Filtrado):")
print(df_equipamentos.head().to_markdown(index=False, numalign="left", stralign="left"))

# Exibir as colunas e seus tipos do DataFrame filtrado
print("\nInformações do DataFrame Equipamentos (Filtrado):")
print(df_equipamentos.info())

DataFrame Equipamentos (Filtrado):
| EQUIPAMENTO   | DENOMINAÇÃO                | POSIÇÃO DO CANHÃO DE INJEÇÃO   | MODELO   | TIPO OBJETO (FAMÍLIA)   | ANO CONSTRUÇÃO   | DATA ENTRADA EM SERVIÇO   |
|:--------------|:---------------------------|:-------------------------------|:---------|:------------------------|:-----------------|:--------------------------|
| IJ-046        | INJETORA DE BORRACHA - REP | SUPERIOR                       | V17EY    | IRP1                    | 2003             | 2011-10-24                |

Informações do DataFrame Equipamentos (Filtrado):
<class 'pandas.core.frame.DataFrame'>
Index: 1 entries, 1 to 1
Data columns (total 7 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   EQUIPAMENTO                   1 non-null      object 
 1   DENOMINAÇÃO                   1 non-null      object 
 2   POSIÇÃO DO CANHÃO DE INJEÇÃO  1 non-null      object 
 3   MODELO                     

In [20]:
# Agora, para construir um modelo preditivo para a quantidade de massa consumida, preciso considerar as variáveis de produção e as informações dos equipamentos."

In [21]:
# Combinar os DataFrames df_equipamentos e df_producao com base nas colunas "EQUIPAMENTO" e "Cód. Recurso"
df_combinado = pd.merge(
    df_equipamentos, df_producao, left_on="EQUIPAMENTO", right_on="Cód. Recurso"
)

# Converter as colunas "Data de Produção" e "DATA ENTRADA EM SERVIÇO" para o tipo datetime
df_combinado["Data de Produção"] = pd.to_datetime(df_combinado["Data de Produção"])
df_combinado["DATA ENTRADA EM SERVIÇO"] = pd.to_datetime(
    df_combinado["DATA ENTRADA EM SERVIÇO"]
)

# Calcular a coluna "Dias em Serviço" como a diferença em dias entre as colunas "Data de Produção" e "DATA ENTRADA EM SERVIÇO"
df_combinado["Dias em Serviço"] = (
    df_combinado["Data de Produção"] - df_combinado["DATA ENTRADA EM SERVIÇO"]
).dt.days

# Calcular a coluna "Consumo de massa total" como o produto das colunas "Qtd. Produzida" e "Consumo de massa no item( em Kg)"
df_combinado["Consumo de massa total"] = (
    df_combinado["Qtd. Produzida"] * df_combinado["Consumo de massa no item( em Kg)"]
)

# Agregar as colunas "Consumo de massa total" e "Qtd. Produzida" pelas colunas especificadas
df_agg = df_combinado.groupby(
    [
        "Cód. Produto",
        "Descrição da massa (Composto)",
        "Dias em Serviço",
        "ANO CONSTRUÇÃO",
        "MODELO",
        "TIPO OBJETO (FAMÍLIA)",
        "POSIÇÃO DO CANHÃO DE INJEÇÃO",
    ]
)[["Consumo de massa total", "Qtd. Produzida"]].sum()

# Renomear as colunas agregadas para "Consumo de massa total" e "Qtd. Produzida"
df_agg = df_agg.rename(
    columns={
        "Consumo de massa total": "Consumo de massa total",
        "Qtd. Produzida": "Qtd. Produzida",
    }
)

# Redefinir o índice do DataFrame agregado
df_agg = df_agg.reset_index()

# Exibir as 5 primeiras linhas do DataFrame agregado
print("DataFrame Agregado:")
print(df_agg.head().to_markdown(index=False, numalign="left", stralign="left"))

# Exibir as colunas e seus tipos do DataFrame agregado
print("\nInformações do DataFrame Agregado:")
print(df_agg.info())

DataFrame Agregado:
| Cód. Produto   | Descrição da massa (Composto)   | Dias em Serviço   | ANO CONSTRUÇÃO   | MODELO   | TIPO OBJETO (FAMÍLIA)   | POSIÇÃO DO CANHÃO DE INJEÇÃO   | Consumo de massa total   | Qtd. Produzida   |
|:---------------|:--------------------------------|:------------------|:-----------------|:---------|:------------------------|:-------------------------------|:-------------------------|:-----------------|
| SA02004        | P212/1                          | 4132              | 2003             | V17EY    | IRP1                    | SUPERIOR                       | 308.14                   | 497              |
| SA02961        | P212/1                          | 4132              | 2003             | V17EY    | IRP1                    | SUPERIOR                       | 607.92                   | 447              |
| SA02961        | P212/1                          | 4133              | 2003             | V17EY    | IRP1                    | SUPERIOR           

In [22]:
# Analisar a relação entre as variáveis e a quantidade de massa consumida.
# Calcular e exibir as estatísticas descritivas de "Consumo de massa total" para cada "Cód. Produto" e "Descrição da massa (Composto)"
print("Estatísticas Descritivas do Consumo de Massa Total:")
print(
    df_agg.groupby(["Cód. Produto", "Descrição da massa (Composto)"])[
        "Consumo de massa total"
    ]
    .describe()
    .to_markdown(numalign="left", stralign="left")
)

Estatísticas Descritivas do Consumo de Massa Total:
|                         | count   | mean    | std     | min    | 25%    | 50%     | 75%     | max     |
|:------------------------|:--------|:--------|:--------|:-------|:-------|:--------|:--------|:--------|
| ('SA02004', 'P212/1')   | 1       | 308.14  | nan     | 308.14 | 308.14 | 308.14  | 308.14  | 308.14  |
| ('SA02961', 'P212/1')   | 6       | 860.88  | 599.269 | 187.68 | 309.06 | 934.32  | 1389.24 | 1467.44 |
| ('SA05780', 'N-142/67') | 2       | 270.908 | 318.05  | 46.013 | 158.46 | 270.908 | 383.356 | 495.803 |


In [23]:
#  Verificar se existem dados suficientes para cada combinação de Cód. Produto e Descrição da massa (Composto).

In [24]:
# Contar as ocorrências de cada combinação de "Cód. Produto" e "Descrição da massa (Composto)"
contagem_combinacoes = df_agg.groupby(
    ["Cód. Produto", "Descrição da massa (Composto)"]
).size()

# Filtrar as combinações com menos de 5 pontos de dados
combinacoes_validas = contagem_combinacoes[contagem_combinacoes >= 5].index
df_agg_filtered = df_agg[
    df_agg[["Cód. Produto", "Descrição da massa (Composto)"]]
    .apply(tuple, axis=1)
    .isin(combinacoes_validas)
]

# Exibir as 5 primeiras linhas do DataFrame filtrado
print("DataFrame Agregado (Filtrado):")
print(df_agg_filtered.head().to_markdown(index=False, numalign="left", stralign="left"))

# Exibir as colunas e seus tipos do DataFrame filtrado
print("\nInformações do DataFrame Agregado (Filtrado):")
print(df_agg_filtered.info())

DataFrame Agregado (Filtrado):
| Cód. Produto   | Descrição da massa (Composto)   | Dias em Serviço   | ANO CONSTRUÇÃO   | MODELO   | TIPO OBJETO (FAMÍLIA)   | POSIÇÃO DO CANHÃO DE INJEÇÃO   | Consumo de massa total   | Qtd. Produzida   |
|:---------------|:--------------------------------|:------------------|:-----------------|:---------|:------------------------|:-------------------------------|:-------------------------|:-----------------|
| SA02961        | P212/1                          | 4132              | 2003             | V17EY    | IRP1                    | SUPERIOR                       | 607.92                   | 447              |
| SA02961        | P212/1                          | 4133              | 2003             | V17EY    | IRP1                    | SUPERIOR                       | 1467.44                  | 1079             |
| SA02961        | P212/1                          | 4134              | 2003             | V17EY    | IRP1                    | SUPERIOR

In [25]:
# Os dados foram filtrados, e podemos prosserguir com a construção do modelo preditivo.

In [30]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Separar as variáveis de entrada (features) e a variável de saída (target)
features = df_agg_filtered[
    [
        "Cód. Produto",
        "Descrição da massa (Composto)",
        "Dias em Serviço",
        "ANO CONSTRUÇÃO",
        "MODELO",
        "TIPO OBJETO (FAMÍLIA)",
        "POSIÇÃO DO CANHÃO DE INJEÇÃO",
    ]
]
target = df_agg_filtered["Consumo de massa total"]

# Aplicar One-Hot Encoding nas variáveis categóricas
encoder = OneHotEncoder(handle_unknown="ignore")
encoded_features = encoder.fit_transform(
    features[
        [
            "Cód. Produto",
            "Descrição da massa (Composto)",
            "MODELO",
            "TIPO OBJETO (FAMÍLIA)",
            "POSIÇÃO DO CANHÃO DE INJEÇÃO",
        ]
    ]
).toarray()

# Combinar as features numéricas com as features codificadas
X = pd.concat(
    [
        pd.DataFrame(encoded_features),
        features[["Dias em Serviço", "ANO CONSTRUÇÃO"]].reset_index(drop=True),
    ],
    axis=1,
)
y = target

# Dividir os dados em conjuntos de treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [32]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Converter todos os nomes das colunas em X_train e X_test para string
X_train.columns = X_train.columns.astype(str)
X_test.columns = X_test.columns.astype(str)

# Criar e treinar o modelo Random Forest Regressor
model = RandomForestRegressor(random_state=42)

# Treinar o modelo com os dados de treinamento
model.fit(X_train, y_train)

# Fazer previsões com os dados de teste
y_pred = model.predict(X_test)

# Exibir as previsões
print(f"Previsões do modelo: {y_pred}")

Previsões do modelo: [1053.2928 1053.2928]


In [33]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Calcular as métricas de erro
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse**0.5
r2 = r2_score(y_test, y_pred)

# Exibir as métricas de erro
print(f"Erro Médio Absoluto (MAE): {mae:.2f}")
print(f"Erro Quadrático Médio (MSE): {mse:.2f}")
print(f"Raiz do Erro Quadrático Médio (RMSE): {rmse:.2f}")
print(f"R-quadrado (R²): {r2:.2f}")

Erro Médio Absoluto (MAE): 429.76
Erro Quadrático Médio (MSE): 184937.42
Raiz do Erro Quadrático Médio (RMSE): 430.04
R-quadrado (R²): -0.00


## Avaliação do Modelo Preditivo

O modelo preditivo construído apresentou as seguintes métricas de erro:

* **Erro Médio Absoluto (MAE):** 429,76
* **Erro Quadrático Médio (MSE):** 184937,42
* **Raiz do Erro Quadrático Médio (RMSE):** 430,04
* **R-quadrado (R²):** -0,00

O valor de R² (-0,00) indica que o modelo **não se ajusta bem aos dados** e não consegue explicar a variabilidade no tempo restante para manutenção. Isso pode ser explicado pela **pequena quantidade de dados disponíveis para treinamento**, especialmente após a filtragem das combinações de produto e massa com menos de 5 pontos de dados, o que resultou em apenas **6 amostras**.

**Observações:**

Com uma quantidade tão pequena de dados, o modelo tem dificuldade em aprender os padrões e generalizar para novas situações. Além disso, a filtragem pode ter removido informações importantes do conjunto de dados.

**Recomendações:**

Para construir um modelo preditivo mais preciso, seria necessário ter acesso a **mais dados de produção**, abrangendo diferentes recursos e com **maior número de amostras para cada combinação de produto e massa**. 

Com mais dados, o modelo teria mais informações para aprender e generalizar, o que poderia levar a uma melhoria significativa nas métricas de erro e no valor de R².

# Analise de Dados e Matriz de Correlacao

In [34]:
# Crie uma cópia de df_agg_filtered
df_encoded = df_agg_filtered.copy()

# Aplique a codificação one-hot nas colunas especificadas
encoder = OneHotEncoder(handle_unknown="ignore")
encoded_features = encoder.fit_transform(
    df_encoded[
        [
            "Cód. Produto",
            "Descrição da massa (Composto)",
            "MODELO",
            "TIPO OBJETO (FAMÍLIA)",
            "POSIÇÃO DO CANHÃO DE INJEÇÃO",
        ]
    ]
).toarray()
df_encoded = pd.concat([df_encoded, pd.DataFrame(encoded_features)], axis=1)

# Remova as colunas originais
df_encoded = df_encoded.drop(
    [
        "Cód. Produto",
        "Descrição da massa (Composto)",
        "MODELO",
        "TIPO OBJETO (FAMÍLIA)",
        "POSIÇÃO DO CANHÃO DE INJEÇÃO",
    ],
    axis=1,
)

# Exiba as 5 primeiras linhas do DataFrame codificado
print("DataFrame Codificado:")
print(df_encoded.head().to_markdown(index=False, numalign="left", stralign="left"))

# Exiba as colunas e seus tipos do DataFrame codificado
print("\nInformações do DataFrame Codificado:")
print(df_encoded.info())

DataFrame Codificado:
| Dias em Serviço   | ANO CONSTRUÇÃO   | Consumo de massa total   | Qtd. Produzida   | 0   | 1   | 2   | 3   | 4   |
|:------------------|:-----------------|:-------------------------|:-----------------|:----|:----|:----|:----|:----|
| 4132              | 2003             | 607.92                   | 447              | 1   | 1   | 1   | 1   | 1   |
| 4133              | 2003             | 1467.44                  | 1079             | 1   | 1   | 1   | 1   | 1   |
| 4134              | 2003             | 1432.08                  | 1053             | 1   | 1   | 1   | 1   | 1   |
| 4136              | 2003             | 209.44                   | 154              | 1   | 1   | 1   | 1   | 1   |
| 4137              | 2003             | 1260.72                  | 927              | 1   | 1   | 1   | 1   | 1   |

Informações do DataFrame Codificado:
<class 'pandas.core.frame.DataFrame'>
Index: 7 entries, 1 to 0
Data columns (total 9 columns):
 #   Column               

In [35]:
import altair as alt

# Calcule a matriz de correlação
corr_matrix = df_encoded.corr()

# Crie um mapa de calor da matriz de correlação
heatmap = (
    alt.Chart(corr_matrix.reset_index().melt("index"))
    .mark_rect()
    .encode(
        x=alt.X("index", title=""),
        y=alt.Y("variable", title=""),
        color=alt.Color("value", scale=alt.Scale(range="heatmap")),
        tooltip=["index", "variable", "value"],
    )
    .properties(title="Matriz de Correlação")
    .interactive()
)

# Salve o gráfico em um arquivo JSON
heatmap.save("correlation_matrix_heatmap.json")

# Exiba a matriz de correlação
print("Matriz de Correlação:")
print(corr_matrix.to_markdown(numalign="left", stralign="left"))

Matriz de Correlação:
|                        | Dias em Serviço   | ANO CONSTRUÇÃO   | Consumo de massa total   | Qtd. Produzida   | 0   | 1   | 2   | 3   | 4   |
|:-----------------------|:------------------|:-----------------|:-------------------------|:-----------------|:----|:----|:----|:----|:----|
| Dias em Serviço        | 1                 | nan              | -0.408538                | -0.408538        | nan | nan | nan | nan | nan |
| ANO CONSTRUÇÃO         | nan               | nan              | nan                      | nan              | nan | nan | nan | nan | nan |
| Consumo de massa total | -0.408538         | nan              | 1                        | 1                | nan | nan | nan | nan | nan |
| Qtd. Produzida         | -0.408538         | nan              | 1                        | 1                | nan | nan | nan | nan | nan |
| 0                      | nan               | nan              | nan                      | nan              | nan | nan | na

In [36]:
heatmap.save("correlation_matrix_heatmap.png")

In [38]:
import altair as alt

# Gráfico de dispersão com reta de regressão linear
scatter_chart = (
    alt.Chart(df_agg_filtered)
    .mark_point()
    .encode(
        x="Dias em Serviço",
        y="Consumo de massa total",
        tooltip=["Dias em Serviço", "Consumo de massa total"],
    )
    .properties(
        title="Gráfico de Dispersão - Dias em Serviço vs. Consumo de massa total"
    )
)

# Ajuste da reta de regressão
line = scatter_chart.transform_regression(
    "Dias em Serviço", "Consumo de massa total"
).mark_line()

# Combinação do gráfico de dispersão e da reta de regressão
scatter_with_line = scatter_chart + line
scatter_with_line = scatter_with_line.interactive()

# Salva o gráfico de dispersão em um arquivo HTML
scatter_with_line.save("dias_em_servico_vs_consumo_massa_total.html")

# Histogramas
histogram_dias_servico = (
    alt.Chart(df_agg_filtered)
    .mark_bar()
    .encode(
        alt.X("Dias em Serviço", bin=True),
        y="count()",
        tooltip=[alt.Tooltip("Dias em Serviço", bin=True), "count()"],
    )
    .properties(title="Histograma - Dias em Serviço")
    .interactive()
)

histogram_consumo_massa = (
    alt.Chart(df_agg_filtered)
    .mark_bar()
    .encode(
        alt.X("Consumo de massa total", bin=True),
        y="count()",
        tooltip=[alt.Tooltip("Consumo de massa total", bin=True), "count()"],
    )
    .properties(title="Histograma - Consumo de massa total")
    .interactive()
)

histogram_qtd_produzida = (
    alt.Chart(df_agg_filtered)
    .mark_bar()
    .encode(
        alt.X("Qtd. Produzida", bin=True),
        y="count()",
        tooltip=[alt.Tooltip("Qtd. Produzida", bin=True), "count()"],
    )
    .properties(title="Histograma - Qtd. Produzida")
    .interactive()
)

# Salva os histogramas em arquivos HTML separados
histogram_dias_servico.save("histograma_dias_em_servico.html")
histogram_consumo_massa.save("histograma_consumo_massa_total.html")
histogram_qtd_produzida.save("histograma_qtd_produzida.html")

# Boxplot
boxplot_consumo_massa = (
    alt.Chart(df_agg_filtered)
    .mark_boxplot()
    .encode(y="Consumo de massa total")
    .properties(title="Boxplot - Consumo de massa total")
)

# Salva o boxplot em um arquivo HTML
boxplot_consumo_massa.save("boxplot_consumo_massa_total.html")

In [39]:
boxplot_consumo_massa.save("boxplot_consumo_massa_total.png")

In [41]:
# Filter df_equipamentos to include only the rows where the column "EQUIPAMENTO" is equal to "IJ-046"
df_equipamentos = df_equipamentos[df_equipamentos["EQUIPAMENTO"] == "IJ-046"]

# Combine the DataFrames df_equipamentos and df_producao based on the columns "EQUIPAMENTO" and "Cód. Recurso"
df_combinado = pd.merge(
    df_equipamentos, df_producao, left_on="EQUIPAMENTO", right_on="Cód. Recurso"
)

# Convert the columns "Data de Produção" and "DATA ENTRADA EM SERVIÇO" to the datetime type
df_combinado["Data de Produção"] = pd.to_datetime(df_combinado["Data de Produção"])
df_combinado["DATA ENTRADA EM SERVIÇO"] = pd.to_datetime(
    df_combinado["DATA ENTRADA EM SERVIÇO"]
)

# Calculate the column "Dias em Serviço" as the difference in days between the columns "Data de Produção" and "DATA ENTRADA EM SERVIÇO"
df_combinado["Dias em Serviço"] = (
    df_combinado["Data de Produção"] - df_combinado["DATA ENTRADA EM SERVIÇO"]
).dt.days

# Calculate the column "Consumo de massa total" as the product of the columns "Qtd. Produzida" and "Consumo de massa no item( em Kg)"
df_combinado["Consumo de massa total"] = (
    df_combinado["Qtd. Produzida"] * df_combinado["Consumo de massa no item( em Kg)"]
)

# Aggregate the columns "Consumo de massa total" and "Qtd. Produzida" by the specified columns
df_agg = df_combinado.groupby(
    [
        "Cód. Produto",
        "Descrição da massa (Composto)",
        "Dias em Serviço",
        "ANO CONSTRUÇÃO",
        "MODELO",
        "TIPO OBJETO (FAMÍLIA)",
        "POSIÇÃO DO CANHÃO DE INJEÇÃO",
    ]
)[["Consumo de massa total", "Qtd. Produzida"]].sum()

# Rename the aggregated columns to "Consumo de massa total" and "Qtd. Produzida"
df_agg = df_agg.rename(
    columns={
        "Consumo de massa total": "Consumo de massa total",
        "Qtd. Produzida": "Qtd. Produzida",
    }
)

# Reset the index of the aggregated DataFrame
df_agg = df_agg.reset_index()

# Count the occurrences of each combination of "Cód. Produto" and "Descrição da massa (Composto)"
contagem_combinacoes = df_agg.groupby(
    ["Cód. Produto", "Descrição da massa (Composto)"]
).size()

# Filter the combinations with less than 5 points of data
combinacoes_validas = contagem_combinacoes[contagem_combinacoes >= 5].index
df_agg_filtered = df_agg[
    df_agg[["Cód. Produto", "Descrição da massa (Composto)"]]
    .apply(tuple, axis=1)
    .isin(combinacoes_validas)
]

# Display the first 5 rows of the filtered aggregated DataFrame
print("DataFrame Agregado (Filtrado):")
print(df_agg_filtered.head().to_markdown(index=False, numalign="left", stralign="left"))

# Display the columns and their types of the filtered aggregated DataFrame
print("\nInformações do DataFrame Agregado (Filtrado):")
print(df_agg_filtered.info())

DataFrame Agregado (Filtrado):
| Cód. Produto   | Descrição da massa (Composto)   | Dias em Serviço   | ANO CONSTRUÇÃO   | MODELO   | TIPO OBJETO (FAMÍLIA)   | POSIÇÃO DO CANHÃO DE INJEÇÃO   | Consumo de massa total   | Qtd. Produzida   |
|:---------------|:--------------------------------|:------------------|:-----------------|:---------|:------------------------|:-------------------------------|:-------------------------|:-----------------|
| SA02961        | P212/1                          | 4132              | 2003             | V17EY    | IRP1                    | SUPERIOR                       | 607.92                   | 447              |
| SA02961        | P212/1                          | 4133              | 2003             | V17EY    | IRP1                    | SUPERIOR                       | 1467.44                  | 1079             |
| SA02961        | P212/1                          | 4134              | 2003             | V17EY    | IRP1                    | SUPERIOR

In [42]:
import altair as alt

# Scatter plot with linear regression line
scatter_chart = (
    alt.Chart(df_agg_filtered)
    .mark_point()
    .encode(
        x="Dias em Serviço",
        y="Consumo de massa total",
        tooltip=["Dias em Serviço", "Consumo de massa total"],
    )
    .properties(
        title="Gráfico de Dispersão - Dias em Serviço vs. Consumo de massa total"
    )
)

# Fit the regression line
line = scatter_chart.transform_regression(
    "Dias em Serviço", "Consumo de massa total"
).mark_line()

# Combine the scatter plot and the regression line
scatter_with_line = scatter_chart + line
scatter_with_line = scatter_with_line.interactive()

# Save the scatter plot to an HTML file
scatter_with_line.save("dias_em_servico_vs_consumo_massa_total.html")

# Histograms
histogram_dias_servico = (
    alt.Chart(df_agg_filtered)
    .mark_bar()
    .encode(
        alt.X("Dias em Serviço", bin=True),
        y="count()",
        tooltip=[alt.Tooltip("Dias em Serviço", bin=True), "count()"],
    )
    .properties(title="Histograma - Dias em Serviço")
    .interactive()
)

histogram_consumo_massa = (
    alt.Chart(df_agg_filtered)
    .mark_bar()
    .encode(
        alt.X("Consumo de massa total", bin=True),
        y="count()",
        tooltip=[alt.Tooltip("Consumo de massa total", bin=True), "count()"],
    )
    .properties(title="Histograma - Consumo de massa total")
    .interactive()
)

histogram_qtd_produzida = (
    alt.Chart(df_agg_filtered)
    .mark_bar()
    .encode(
        alt.X("Qtd. Produzida", bin=True),
        y="count()",
        tooltip=[alt.Tooltip("Qtd. Produzida", bin=True), "count()"],
    )
    .properties(title="Histograma - Qtd. Produzida")
    .interactive()
)

# Save the histograms to separate HTML files
histogram_dias_servico.save("histograma_dias_em_servico.html")
histogram_consumo_massa.save("histograma_consumo_massa_total.html")
histogram_qtd_produzida.save("histograma_qtd_produzida.html")

# Boxplot
boxplot_consumo_massa = (
    alt.Chart(df_agg_filtered)
    .mark_boxplot()
    .encode(y="Consumo de massa total")
    .properties(title="Boxplot - Consumo de massa total")
)

# Save the boxplot to an HTML file
boxplot_consumo_massa.save("boxplot_consumo_massa_total.html")

## Análise Exploratória dos Dados: Interpretação dos Gráficos

Os gráficos gerados fornecem informações valiosas sobre a distribuição e a relação entre as variáveis:

* **Gráfico de Dispersão:** O gráfico de dispersão entre `Dias em Serviço` e `Consumo de massa total` indica uma **tendência decrescente**, o que sugere que o consumo de massa tende a diminuir com o tempo de uso do equipamento. Essa tendência pode ser um indicativo de que os equipamentos se tornam mais eficientes no uso da massa ao longo do tempo, ou que o desgaste natural leva a uma redução na necessidade de massa.

* **Histogramas:** Os histogramas mostram a **distribuição de cada variável**, o que pode ajudar a identificar outliers (valores discrepantes) e padrões nos dados. Por exemplo, um histograma com uma distribuição assimétrica pode indicar a presença de outliers ou a necessidade de transformação da variável.

* **Boxplot:** O boxplot do `Consumo de massa total` mostra a mediana, os quartis e os outliers da variável, o que ajuda a entender a variabilidade e a presença de valores extremos.  A partir do boxplot, podemos identificar a presença de outliers e a dispersão dos dados em torno da mediana.

### Importância da Análise Exploratória

A análise exploratória dos dados é um passo **crucial** para a construção de modelos preditivos mais precisos. A análise gráfica, em particular, é uma ferramenta importante para entender melhor os dados e as relações entre as variáveis.

### Limitações e Próximos Passos

Vale ressaltar que a quantidade de dados analisada ainda é **pequena**, o que pode **limitar a confiabilidade das conclusões obtidas**. A obtenção de mais dados, abrangendo diferentes recursos e com maior número de amostras para cada combinação de produto e massa, seria fundamental para uma análise mais robusta e para a construção de um modelo preditivo mais preciso.